# NB10 — AutoGluon Model Inspection & Hyperparameter Reasoning


---

## Objective

This notebook does **not** retrain AutoGluon and does **not** repeat the benchmark from NB9.

The goal is to inspect the trained AutoGluon predictors generated in NB9 in order to understand:

- which models AutoGluon actually trained
- which hyperparameters were used
- how LightGBM differs from LightGBMXT
- how the Weighted Ensemble combines base models
- whether the best AutoGluon model is suitable as a production candidate

NB10 focuses on:

- model inspection
- hyperparameter reasoning
- production-oriented model selection

---

## Context

NB9 showed that AutoGluon improved the manual LightGBM benchmark.

### Reference benchmark results

| Model | RMSLE | MAE | RMSE | R² |
|---|---:|---:|---:|---:|
| Manual LightGBM (NB5) | 0.5999 | 68.45 | 248.71 | 0.9651 |
| AutoGluon (NB9) | 0.4761 | 55.54 | 201.04 | 0.9760 |

However, leaderboard performance alone is not enough.

In this notebook we inspect:

- predictor metadata
- leaderboard internals
- trained model family
- real hyperparameters
- ensemble composition
- production trade-offs

---

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 300)

print("Setup OK")

Setup OK


---

## Load AutoGluon Predictors (Robust Loading)

We load the AutoGluon predictors trained in NB9.

Important:

- this notebook does not retrain models
- this notebook only inspects saved predictors
- corrupted predictors are skipped automatically
- inspection continues using valid folds only

In [2]:
PROJECT_ROOT = Path("..").resolve()

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "nb9_autogluon_tabular"
AUTOGLUON_DIR = ARTIFACTS_DIR / "autogluon_models"

FOLD_PATHS = {
    1: AUTOGLUON_DIR / "fold_1",
    2: AUTOGLUON_DIR / "fold_2",
}

for fold, path in FOLD_PATHS.items():
    print(f"Fold {fold} path:", path.resolve())
    print("Exists:", path.exists())

Fold 1 path: /home/donatocorbacio/projects/store-sales-project/artifacts/nb9_autogluon_tabular/autogluon_models/fold_1
Exists: True
Fold 2 path: /home/donatocorbacio/projects/store-sales-project/artifacts/nb9_autogluon_tabular/autogluon_models/fold_2
Exists: True


In [3]:
predictors = {}
failed_folds = {}

for fold, path in FOLD_PATHS.items():
    if not path.exists():
        failed_folds[fold] = "Path does not exist"
        print(f"Skipped Fold {fold}: path not found")
        continue
    
    try:
        predictors[fold] = TabularPredictor.load(str(path))
        print(f"Fold {fold} predictor loaded.")
    except Exception as e:
        failed_folds[fold] = f"{type(e).__name__}: {e}"
        print(f"Skipped Fold {fold}: {type(e).__name__}: {e}")

Fold 1 predictor loaded.
Fold 2 predictor loaded.


In [4]:
if not predictors:
    raise RuntimeError("No valid AutoGluon predictors found. Check NB9 artifacts.")

print("Loaded folds:", list(predictors.keys()))

if failed_folds:
    print("\nSkipped folds:")
    for fold, reason in failed_folds.items():
        print(f"- Fold {fold}: {reason}")

Loaded folds: [1, 2]


---

## Basic Predictor Inspection

Before inspecting the models, we check the main predictor metadata:

- label
- problem type
- evaluation metric
- saved path

In [5]:
for fold, predictor in predictors.items():
    print("=" * 80)
    print(f"FOLD {fold}")
    print("=" * 80)
    print("Path:", predictor.path)
    print("Label:", predictor.label)
    print("Problem type:", predictor.problem_type)
    print("Evaluation metric:", predictor.eval_metric)

FOLD 1
Path: /home/donatocorbacio/projects/store-sales-project/artifacts/nb9_autogluon_tabular/autogluon_models/fold_1
Label: sales
Problem type: regression
Evaluation metric: root_mean_squared_error
FOLD 2
Path: /home/donatocorbacio/projects/store-sales-project/artifacts/nb9_autogluon_tabular/autogluon_models/fold_2
Label: sales
Problem type: regression
Evaluation metric: root_mean_squared_error


---

## Leaderboard Inspection

The leaderboard gives a performance ranking of the trained models.

Important:

The leaderboard is useful, but it does not fully explain model behavior.

For that reason, this notebook goes deeper using:

- `predictor.info()`
- `predictor.fit_summary()`
- `predictor.model_hyperparameters()`
- `predictor.model_info()`

In [6]:
leaderboards = []

for fold, predictor in predictors.items():
    leaderboard = predictor.leaderboard(silent=True, extra_info=True)
    leaderboard["fold"] = fold
    leaderboards.append(leaderboard)

leaderboard_all = pd.concat(leaderboards, ignore_index=True)

leaderboard_all[
    [
        "fold",
        "model",
        "score_val",
        "eval_metric",
        "pred_time_val",
        "fit_time",
        "stack_level",
        "can_infer",
    ]
].sort_values(["fold", "score_val"], ascending=[True, False])

,fold,model,score_val,eval_metric,pred_time_val,fit_time,stack_level,can_infer
0,2,WeightedEnsemble_L2,-156.877041,root_mean_squared_error,7.493559,587.524698,2.0,1.0
1,2,LightGBMXT,-157.046819,root_mean_squared_error,7.434825,566.318337,1.0,1.0
2,2,LightGBM,-181.277652,root_mean_squared_error,0.058172,21.192311,1.0,1.0


---

## Best Model per Fold

We explicitly extract the best model selected by AutoGluon for each fold.

In [8]:
best_models = {}
invalid_folds = {}

for fold, predictor in predictors.items():
    try:
        model_names = predictor.model_names(can_infer=True)
        
        if len(model_names) == 0:
            invalid_folds[fold] = "No models available for inference"
            print(f"Fold {fold} skipped: no inferable models")
            continue

        best_model = predictor.model_best
        best_models[fold] = best_model
        print(f"Fold {fold} best model:", best_model)

    except Exception as e:
        invalid_folds[fold] = f"{type(e).__name__}: {e}"
        print(f"Fold {fold} skipped:", invalid_folds[fold])

valid_predictors = {
    fold: predictor
    for fold, predictor in predictors.items()
    if fold not in invalid_folds
}

print("Valid folds for inspection:", list(valid_predictors.keys()))

Fold 1 skipped: no inferable models
Fold 2 best model: WeightedEnsemble_L2
Valid folds for inspection: [2]


---

## Trained Models

Now we inspect which models were actually trained.

This matters because AutoGluon may define many candidate models, but only a subset is actually trained.

In [9]:
for fold, predictor in predictors.items():
    print("=" * 80)
    print(f"FOLD {fold} - MODEL NAMES")
    print("=" * 80)
    
    for model_name in predictor.model_names():
        print("-", model_name)

FOLD 1 - MODEL NAMES
FOLD 2 - MODEL NAMES
- LightGBMXT
- LightGBM
- WeightedEnsemble_L2


---

## Fit Summary

`fit_summary()` provides a compact overview of the AutoGluon training result.

In [10]:
fit_summaries = {}

for fold, predictor in predictors.items():
    print("=" * 80)
    print(f"FOLD {fold} - FIT SUMMARY KEYS")
    print("=" * 80)
    
    summary = predictor.fit_summary(verbosity=0, show_plot=False)
    fit_summaries[fold] = summary
    
    print(summary.keys())

FOLD 1 - FIT SUMMARY KEYS
dict_keys(['model_types', 'model_performance', 'model_best', 'model_paths', 'model_fit_times', 'model_pred_times', 'num_bag_folds', 'max_stack_level', 'model_hyperparams'])
FOLD 2 - FIT SUMMARY KEYS
dict_keys(['model_types', 'model_performance', 'model_best', 'model_paths', 'model_fit_times', 'model_pred_times', 'num_bag_folds', 'max_stack_level', 'model_hyperparams'])


In [11]:
for fold, summary in fit_summaries.items():
    print("=" * 80)
    print(f"FOLD {fold}")
    print("=" * 80)
    
    for key, value in summary.items():
        if key in ["model_types", "model_performance", "model_best"]:
            print(f"\n{key}:")
            print(value)

FOLD 1

model_types:
{}

model_performance:
{}

model_best:
None
FOLD 2

model_types:
{'LightGBMXT': 'LGBModel', 'LightGBM': 'LGBModel', 'WeightedEnsemble_L2': 'WeightedEnsembleModel'}

model_performance:
{'LightGBMXT': np.float64(-157.04681859301277), 'LightGBM': np.float64(-181.27765170391152), 'WeightedEnsemble_L2': np.float64(-156.87704051921412)}

model_best:
WeightedEnsemble_L2


---

## Predictor Internal Info

`predictor.info()` exposes internal predictor metadata.

Useful for technical inspection, not for production logic.

---

## Hyperparameter Inspection

This is the core section of NB10.

We inspect the real hyperparameters used by each trained model.

In [12]:
def pretty_print_dict(d, max_chars=4000):
    text = json.dumps(d, indent=4, default=str)
    if len(text) > max_chars:
        print(text[:max_chars])
        print("\n... [truncated]")
    else:
        print(text)

In [13]:
def inspect_model_hyperparameters(predictor, model_name):
    print("=" * 100)
    print(f"MODEL: {model_name}")
    print("=" * 100)

    try:
        user_params = predictor.model_hyperparameters(
            model=model_name,
            output_format="user"
        )
        print("\nUSER / NON-DEFAULT HYPERPARAMETERS:")
        pretty_print_dict(user_params)
    except Exception as e:
        print("Could not retrieve user hyperparameters:", e)

    try:
        all_params = predictor.model_hyperparameters(
            model=model_name,
            output_format="all"
        )
        print("\nALL HYPERPARAMETERS:")
        pretty_print_dict(all_params)
    except Exception as e:
        print("Could not retrieve all hyperparameters:", e)

In [15]:
for fold, predictor in valid_predictors.items():
    print("\n" + "#" * 120)
    print(f"FOLD {fold}")
    print("#" * 120)

    for model_name in predictor.model_names():
        inspect_model_hyperparameters(predictor, model_name)


########################################################################################################################
FOLD 2
########################################################################################################################
MODEL: LightGBMXT

USER / NON-DEFAULT HYPERPARAMETERS:
{
    "extra_trees": true
}

ALL HYPERPARAMETERS:
{
    "learning_rate": 0.05,
    "extra_trees": true,
    "seed": 0
}
MODEL: LightGBM

USER / NON-DEFAULT HYPERPARAMETERS:
{}

ALL HYPERPARAMETERS:
{
    "learning_rate": 0.05,
    "seed": 0
}
MODEL: WeightedEnsemble_L2

USER / NON-DEFAULT HYPERPARAMETERS:
{
    "ag_args_ensemble": {
        "save_bag_folds": true
    }
}

ALL HYPERPARAMETERS:
{
    "ensemble_size": 25,
    "subsample_size": 1000000,
    "ag_args_fit": {
        "max_memory_usage_ratio": 1.0,
        "max_time_limit_ratio": 1.0,
        "max_time_limit": null,
        "min_time_limit": 0,
        "valid_raw_types": null,
        "valid_special_types": null,
        "igno

### Runtime note

During notebook reconstruction, Fold 1 was detected as not inferable.

The predictor folder exists on disk, but AutoGluon reports that no fitted models are available for inference.

For this reason, the runtime inspection continues only with the valid predictor from Fold 2.

The original complete NB10 PDF remains the reference document for the full two-fold inspection.

## LightGBM Booster Inspection

AutoGluon reports high-level hyperparameters for each trained model.

For LightGBM-based models, we also inspect the underlying booster to understand:

- number of boosting rounds
- number of features
- feature importance by gain
- feature importance by split frequency

The goal is not to manually inspect every individual tree, but to understand which variables drive the model and whether the AutoML result is technically explainable.

In [16]:
def inspect_lgbm_booster(predictor, model_name):
    print("=" * 100)
    print(f"BOOSTER INSPECTION: {model_name}")
    print("=" * 100)

    try:
        model = predictor._trainer.load_model(model_name)
        booster = model.model

        print("AutoGluon model class:", type(model))
        print("Underlying booster class:", type(booster))
        print("Number of trees / boosting rounds:", booster.num_trees())
        print("Number of features:", booster.num_feature())

        importance_gain = booster.feature_importance(importance_type="gain")
        importance_split = booster.feature_importance(importance_type="split")
        feature_names = booster.feature_name()

        importance_df = (
            pd.DataFrame({
                "feature": feature_names,
                "importance_gain": importance_gain,
                "importance_split": importance_split,
            })
            .sort_values("importance_gain", ascending=False)
            .reset_index(drop=True)
        )

        display(importance_df)

        return importance_df

    except Exception as e:
        print(f"Could not inspect booster for {model_name}: {type(e).__name__}: {e}")
        return None

In [17]:
booster_importances = {}

for fold, predictor in valid_predictors.items():
    print("\n" + "#" * 120)
    print(f"FOLD {fold} - LIGHTGBM BOOSTER INSPECTION")
    print("#" * 120)

    for model_name in predictor.model_names():
        if "LightGBM" in model_name or "GBM" in model_name:
            importance_df = inspect_lgbm_booster(predictor, model_name)
            booster_importances[(fold, model_name)] = importance_df


########################################################################################################################
FOLD 2 - LIGHTGBM BOOSTER INSPECTION
########################################################################################################################
BOOSTER INSPECTION: LightGBMXT
AutoGluon model class: <class 'autogluon.tabular.models.lgb.lgb_model.LGBModel'>
Underlying booster class: <class 'lightgbm.basic.Booster'>
Number of trees / boosting rounds: 9923
Number of features: 16


,feature,importance_gain,importance_split
0,lag_1,9.985786e+12,13576
1,lag_7,8.685781e+12,13317
2,rolling_std_7,4.394726e+12,3391
3,rolling_mean_14,3.907087e+12,4054
4,rolling_mean_7,3.673518e+12,4313
5,store_nbr,9.170062e+11,33214
6,is_weekend,7.808284e+11,20989
7,promo_last_7,6.026312e+11,29355
8,trend_1_7,4.920281e+11,34329
9,dayofweek,4.047986e+11,32127


BOOSTER INSPECTION: LightGBM
AutoGluon model class: <class 'autogluon.tabular.models.lgb.lgb_model.LGBModel'>
Underlying booster class: <class 'lightgbm.basic.Booster'>
Number of trees / boosting rounds: 296
Number of features: 16


,feature,importance_gain,importance_split
0,rolling_mean_7,1.715651e+13,421
1,lag_7,9.754097e+12,754
2,lag_1,5.923786e+12,761
3,dayofweek,4.695915e+11,877
4,day,2.944404e+11,1332
5,rolling_mean_14,2.701102e+11,309
6,onpromotion,1.636787e+11,540
7,trend_1_7,1.388528e+11,692
8,weekofyear,1.320228e+11,817
9,store_nbr,7.128803e+10,625


## Build Hyperparameter Table

To compare AutoGluon models in a compact and production-friendly format, we build a structured hyperparameter table.

This table makes it easier to compare:

- which models were trained
- whether the model was selected as best candidate
- user-defined vs full hyperparameter configuration
- model-level configuration differences

In [18]:
hyperparameter_rows = []

for fold, predictor in valid_predictors.items():
    for model_name in predictor.model_names():
        try:
            user_params = predictor.model_hyperparameters(
                model=model_name,
                output_format="user"
            )
        except Exception:
            user_params = None

        try:
            all_params = predictor.model_hyperparameters(
                model=model_name,
                output_format="all"
            )
        except Exception:
            all_params = None

        hyperparameter_rows.append({
            "fold": fold,
            "model": model_name,
            "is_best_model": model_name == predictor.model_best,
            "user_hyperparameters": user_params,
            "all_hyperparameters": all_params,
        })

hyperparameters_df = pd.DataFrame(hyperparameter_rows)
hyperparameters_df

,fold,model,is_best_model,user_hyperparameters,all_hyperparameters
0,2,LightGBMXT,False,{'extra_trees': True},"{'learning_rate': 0.05, 'extra_trees': True, 'seed': 0}"
1,2,LightGBM,False,{},"{'learning_rate': 0.05, 'seed': 0}"
2,2,WeightedEnsemble_L2,True,{'ag_args_ensemble': {'save_bag_folds': True}},"{'ensemble_size': 25, 'subsample_size': 1000000, 'ag_args_fit': {'max_memory_usage_ratio': 1.0, 'max_time_limit_ratio': 1.0, 'max_time_limit': None, 'min_time_limit': 0, 'valid_raw_types': None, 'valid_special_types': None, 'ignored_type_group_special': None, 'ignored_type_group_raw': None, 'get..."


## Model Info Inspection

`model_info()` provides model-level metadata beyond leaderboard score.

This is useful in a production-oriented setting because it exposes:

- validation score
- fit time
- prediction time
- model type
- stack level
- inference availability
- number of input features

In [19]:
def inspect_model_info(predictor, model_name):
    print("=" * 100)
    print(f"MODEL INFO: {model_name}")
    print("=" * 100)

    try:
        info = predictor.model_info(model_name)

        for key, value in info.items():
            print(f"\n{key}:")
            if isinstance(value, (dict, list)):
                pretty_print_dict(value)
            else:
                print(value)

        return info

    except Exception as e:
        print("Could not retrieve model info:", e)
        return None

In [20]:
model_info_rows = []

for fold, predictor in valid_predictors.items():
    print("\n" + "#" * 120)
    print(f"FOLD {fold}")
    print("#" * 120)

    for model_name in predictor.model_names():
        info = inspect_model_info(predictor, model_name)

        if info is not None:
            model_info_rows.append({
                "fold": fold,
                "model": model_name,
                "is_best_model": model_name == predictor.model_best,
                "model_type": info.get("model_type"),
                "problem_type": info.get("problem_type"),
                "eval_metric": info.get("eval_metric"),
                "val_score": info.get("val_score"),
                "fit_time": info.get("fit_time"),
                "predict_time": info.get("predict_time"),
                "stack_level": info.get("stack_level"),
                "can_infer": info.get("can_infer"),
                "num_features": len(info.get("features", [])) if info.get("features") is not None else None,
            })

model_info_df = pd.DataFrame(model_info_rows)
model_info_df


########################################################################################################################
FOLD 2
########################################################################################################################
MODEL INFO: LightGBMXT

name:
LightGBMXT

model_type:
LGBModel

problem_type:
regression

eval_metric:
root_mean_squared_error

stopping_metric:
root_mean_squared_error

fit_time:
566.3183369636536

num_classes:
None

quantile_levels:
None

predict_time:
7.434825420379639

val_score:
-157.04681859301277

hyperparameters:
{
    "learning_rate": 0.05,
    "extra_trees": true,
    "seed": 0
}

hyperparameters_user:
{
    "extra_trees": true
}

hyperparameters_fit:
{
    "num_boost_round": 9923
}

hyperparameters_nondefault:
[
    "extra_trees"
]

ag_args_fit:
{
    "max_memory_usage_ratio": 1.0,
    "max_time_limit_ratio": 1.0,
    "max_time_limit": null,
    "min_time_limit": 0,
    "valid_raw_types": [
        "bool",
        "int",
        

,fold,model,is_best_model,model_type,problem_type,eval_metric,val_score,fit_time,predict_time,stack_level,can_infer,num_features
0,2,LightGBMXT,False,LGBModel,regression,root_mean_squared_error,-157.046819,566.318337,7.434825,None,True,16
1,2,LightGBM,False,LGBModel,regression,root_mean_squared_error,-181.277652,21.192311,0.058172,None,True,16
2,2,WeightedEnsemble_L2,True,WeightedEnsembleModel,regression,root_mean_squared_error,-156.877041,0.014050,0.000561,None,True,2


## Weighted Ensemble Inspection

The best AutoGluon model is `WeightedEnsemble_L2`.

To understand why it wins, we inspect the ensemble composition and the weights assigned to its base models.

This is one of the most important sections in the notebook because it explains whether AutoGluon discovered a truly different model or simply combined strong existing candidates more effectively.

In [21]:
ensemble_models = []

for fold, predictor in valid_predictors.items():
    for model_name in predictor.model_names():
        if "WeightedEnsemble" in model_name:
            ensemble_models.append((fold, model_name))

ensemble_models

[(2, 'WeightedEnsemble_L2')]

In [22]:
for fold, model_name in ensemble_models:
    print("\n" + "#" * 120)
    print(f"FOLD {fold} - {model_name}")
    print("#" * 120)

    predictor = valid_predictors[fold]
    info = predictor.model_info(model_name)

    for key, value in info.items():
        if "weight" in key.lower() or "child" in key.lower():
            print(f"\n{key}:")
            if isinstance(value, (dict, list)):
                pretty_print_dict(value)
            else:
                print(value)


########################################################################################################################
FOLD 2 - WeightedEnsemble_L2
########################################################################################################################

children_info:
{
    "S1F1": {
        "name": "S1F1",
        "model_type": "GreedyWeightedEnsembleModel",
        "problem_type": "regression",
        "eval_metric": "root_mean_squared_error",
        "stopping_metric": "root_mean_squared_error",
        "fit_time": 0.014050006866455078,
        "num_classes": null,
        "quantile_levels": null,
        "predict_time": null,
        "val_score": null,
        "hyperparameters": {
            "ensemble_size": 25,
            "subsample_size": 1000000
        },
        "hyperparameters_user": {},
        "hyperparameters_fit": {
            "ensemble_size": 13
        },
        "hyperparameters_nondefault": [],
        "ag_args_fit": {
            "max_memory_us

## Best Ensemble vs Best Single Model

The best leaderboard model is not automatically the best production model.

For that reason, we compare:

- the best overall AutoGluon candidate
- the best single non-ensemble model

This helps evaluate the trade-off between:

- maximum predictive performance
- simplicity and maintainability

In [23]:
candidate_rows = []

for fold, predictor in valid_predictors.items():
    leaderboard = predictor.leaderboard(silent=True, extra_info=True)
    leaderboard = leaderboard.sort_values("score_val", ascending=False).copy()

    best_overall = leaderboard.iloc[0]["model"]

    non_ensemble = leaderboard[
        ~leaderboard["model"].str.contains("WeightedEnsemble", case=False, regex=False)
    ]
    best_single = non_ensemble.iloc[0]["model"]

    candidate_rows.append({
        "fold": fold,
        "candidate_type": "best_overall",
        "model": best_overall,
    })

    candidate_rows.append({
        "fold": fold,
        "candidate_type": "best_single_model",
        "model": best_single,
    })

candidate_df = pd.DataFrame(candidate_rows)
candidate_df

,fold,candidate_type,model
0,2,best_overall,WeightedEnsemble_L2
1,2,best_single_model,LightGBMXT


In [24]:
candidate_comparison = candidate_df.merge(
    leaderboard_all,
    on=["fold", "model"],
    how="left"
)

candidate_comparison[
    [
        "fold",
        "candidate_type",
        "model",
        "score_val",
        "eval_metric",
        "pred_time_val",
        "fit_time",
        "stack_level",
        "can_infer",
    ]
].sort_values(["fold", "candidate_type"])

,fold,candidate_type,model,score_val,eval_metric,pred_time_val,fit_time,stack_level,can_infer
0,2,best_overall,WeightedEnsemble_L2,-156.877041,root_mean_squared_error,7.493559,587.524698,2.0,1.0
1,2,best_single_model,LightGBMXT,-157.046819,root_mean_squared_error,7.434825,566.318337,1.0,1.0


## Model Selection Table

To support production-oriented model selection, we summarize the final candidate comparison in a compact decision table.

This table is designed for practical model selection and balances:

- validation performance
- training cost
- inference cost
- interpretability
- deployment complexity
- operational risk

This is the final decision-oriented view of the AutoML benchmark.

In [27]:
selection_rows = [
    {
        "candidate": "WeightedEnsemble_L2",
        "val_score_rmse": -156.877041,
        "fit_time_sec": 587.524698,
        "predict_time_sec": 7.493559,
        "complexity": "High",
        "interpretability": "Medium",
        "deployment_risk": "Medium-High",
        "recommended_for": "Maximum predictive performance",
        "recommended": "Yes (benchmark winner)"
    },
    {
        "candidate": "LightGBMXT",
        "val_score_rmse": -157.046819,
        "fit_time_sec": 566.318337,
        "predict_time_sec": 7.434825,
        "complexity": "Medium",
        "interpretability": "Medium",
        "deployment_risk": "Medium",
        "recommended_for": "Best single-model production candidate",
        "recommended": "Yes (production candidate)"
    },
    {
        "candidate": "LightGBM",
        "val_score_rmse": -181.277652,
        "fit_time_sec": 21.192311,
        "predict_time_sec": 0.058172,
        "complexity": "Low",
        "interpretability": "High",
        "deployment_risk": "Low",
        "recommended_for": "Fastest baseline / low-cost deployment",
        "recommended": "Fallback only"
    }
]

model_selection_df = pd.DataFrame(selection_rows)
model_selection_df

## Production Decision Matrix

### Option A — WeightedEnsemble_L2

Use when priority is maximum predictive performance.

**Strengths**
- best validation score
- strongest benchmark result
- combines multiple base models

**Weaknesses**
- less interpretable
- more complex deployment
- higher inference cost
- harder debugging

---

### Option B — Best Single LightGBM Model

Use when priority is simplicity and maintainability.

**Strengths**
- easier to deploy
- easier to debug
- simpler monitoring
- faster inference

**Weaknesses**
- slightly lower accuracy than ensemble

## Save Inspection Artifacts

In [25]:
OUTPUT_DIR = Path("../artifacts/nb10_autogluon_model_inspection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_selection_df.to_csv(OUTPUT_DIR / "nb10_model_selection_table.csv", index=False)

leaderboard_all.to_csv(OUTPUT_DIR / "nb10_leaderboard_all.csv", index=False)
hyperparameters_df.to_csv(OUTPUT_DIR / "nb10_hyperparameters_raw.csv", index=False)
model_info_df.to_csv(OUTPUT_DIR / "nb10_model_info.csv", index=False)
candidate_comparison.to_csv(OUTPUT_DIR / "nb10_candidate_comparison.csv", index=False)

print("Saved inspection artifacts to:", OUTPUT_DIR.resolve())

Saved inspection artifacts to: /home/donatocorbacio/projects/store-sales-project/artifacts/nb10_autogluon_model_inspection


## Final Conclusion

NB10 confirms that AutoGluon should not be treated as a black box.

The benchmark gain observed in NB9 does not come from a fundamentally different model family.

Instead, AutoGluon improves performance by:

- leveraging LightGBM-based tree models
- using LightGBMXT as a stronger tree variant
- combining strong base learners through weighted ensembling

The booster inspection confirms that AutoGluon remains centered on gradient-boosted tree models.

Instead of manually inspecting individual trees, the analysis focuses on booster-level properties such as:

- boosting rounds
- feature importance
- split frequency
- model composition

This is more useful in a production-oriented workflow because it explains:

- which variables drive predictions
- why the ensemble performs better
- which model is easier to operationalize

### Engineering Conclusion

- `WeightedEnsemble_L2` is the strongest benchmark candidate
- `LightGBMXT` is the strongest single-model candidate
- AutoML is valuable as a model discovery and benchmarking tool
- final production choice should depend on deployment complexity, monitoring constraints and inference cost

The real value of AutoML in an enterprise workflow is not blind automation.

Its real value is exposing stronger model configurations that can later be audited, controlled and productionized.